In [95]:
import pandas as pd
DATA_DIR = './data'
OUTPUT_DIR = './cleaned_data'


COLUMNS = {
    'main_units': [
        'unit', 'land_unit', 'caste', 'weight', 'tier', 'num_men',
        'recruitment_cost', 'upkeep_cost', 'multiplayer_cost', 'barrier_health',
        'is_high_threat', 'is_monstrous', 'is_renown', 'can_siege',
    ],
    # armour and shield refers to its own table that radious already changes, dont need to predict a specific armour value
    # but rather an armour name 
    'land_units' : [
        'key', 'accuracy', 'armour', 'category', 'charge_bonus', 'class',
        'man_entity', 'melee_attack', 'melee_defence', 'morale', 'bonus_hit_points',
        'num_mounts', 'primary_melee_weapon', 'primary_missile_weapon',
        'shield', 'secondary_ammo', 'primary_ammo', 'damage_mod_flame',
        'damage_mod_magic', 'num_engines', 'damage_mod_physical', 'damage_mod_missile',
        'damage_mod_all', 'can_skirmish', 'can_brace', 'reload', 
    ],
    'battle_entities': [
        'key', 'type', 'walk_speed', 'run_speed', 'acceleration', 'deceleration',
        'charge_speed', 'radius', 'mass', 'turn_speed', 'hit_points', 'fly_speed',
        'flying_charge_speed', 'size',
        
    ],
    'melee_weapons': [
        'key', 'bonus_v_large', 'bonus_v_infantry', 'damage', 'ap_damage', 
        'melee_weapon_type', 'splash_attack_target_size', 'splash_attack_max_attacks',
        'splash_attack_power_multiplier', 'is_magical', 'melee_attack_interval', 
    ],
    'missile_weapons': ['key', 'default_projectile'],
    'projectiles': ['key', 'effective_range', 'minimum_range', 'damage', 'ap_damage', 
                    'base_reload_time', 'bonus_v_infantry', 'bonus_v_large'],
}

In [117]:
def read_table(table, prefix='', data_dir=DATA_DIR):
    """
    Reads a single table selecting useful columns
    prefix + table must match the name of a file in data_dir
    """
    # names = [table]
    # if table == 'main_units':
    path = f"{data_dir}/{prefix}{table}.tsv"
    
    df = pd.read_csv(path, sep='\t', encoding='utf-8')

    # only keep columns defined in the schema
    df_filtered = df.filter(items=COLUMNS[table])
    
    key = 'unit' if table == 'main_units' else 'key'
    df_filtered.drop_duplicates(key, inplace=True)
       
    # rename all columns to include table name to allow merging later on, dont change unit name
    return df_filtered.rename(columns={col: f'{table}__{col}' for col in df_filtered.columns if col != 'unit'})

        

def group_tables(prefix='', data_dir=DATA_DIR):
    """Joins the 5 tables for a given prefix, vanilla or radious part x"""
    # tables holds the 5 tables as seperate entries
    tables = {}
    for table in COLUMNS:       
        tables[table] = read_table(table, prefix, data_dir)
        
    units = pd.merge(tables['main_units'], tables['land_units'], how='left', left_on='main_units__land_unit', right_on='land_units__key')
    
    # formatted table_name, tables key name in left table
    # ie the table battle_entities has the column name of man_entity in land_units
    # perform a left merge onto units
    relations = [
        ('battle_entities', 'land_units__man_entity'),
        ('melee_weapons', 'land_units__primary_melee_weapon'),
        ('missile_weapons', 'land_units__primary_missile_weapon'),
        ('projectiles', 'missile_weapons__default_projectile')    
    ]
    
    for table, column_name in relations:
        units = pd.merge(left=units, right=tables[table], how='left', left_on=column_name, right_on=f'{table}__key')
   
    return units

pairs = pd.DataFrame(columns=["vanilla", "radious"])
unpaired_vanilla = pd.DataFrame(columns=["unit_data"])
unpaired_radious = pd.DataFrame(columns=["unit_data"])

def pair_units(vanilla_units, radious_units):
    global pairs, unpaired_radious, unpaired_vanilla
    for vanilla_unit in vanilla_units.itertuples(index=False):       
        unit_key = vanilla_unit.unit
        unpaired_vanilla.loc[unit_key] = [vanilla_unit._asdict()]
        
    for radious_unit in radious_units.itertuples(index=False):
        radious_unit_key = radious_unit.unit
        
        if radious_unit_key in unpaired_vanilla.index:
            vanilla_unit = unpaired_vanilla.at[radious_unit_key, "unit_data"]
            unpaired_vanilla = unpaired_vanilla.drop(radious_unit_key)

            pairs.loc[radious_unit_key] = [vanilla_unit, radious_unit._asdict()]
        else:
            unpaired_radious.loc[radious_unit_key] = [radious_unit._asdict()]

        

def prepare_data(data_dir=DATA_DIR, output_dir=OUTPUT_DIR):
    vanilla_units = group_tables(data_dir=data_dir)
    
    # merged_radious = pd.DataFrame()  
    # for part in (1,2,3):
    #     # radious_units = group_tables(f'radious_{part}_', data_dir)
    #     radious_units = group_tables(f'grouped_', data_dir)
    #     merged_radious = pd.concat([merged_radious, radious_units], ignore_index=True)
    # merged_radious = pd.concat([merged_radious, radious_units], ignore_index=True)
    # merged_radious =  merged_radious.drop_duplicates()
        
    radious_units = group_tables(f'grouped_', data_dir)
    

    pair_units(vanilla_units, radious_units)
    
    vanilla_units.to_json('./cleaned_data/vanilla.json', orient="records", indent=2)
    pairs.to_json('./cleaned_data/pairs.json', orient="records", indent=2)
    unpaired_radious.to_json('./cleaned_data/unpaired_radious.json', orient="records", indent=2)
    
 

In [118]:
if __name__ == "__main__":
    prepare_data()
    
    